In [1]:
import numpy as np

In [7]:
from typing import NamedTuple

In [8]:
Edge = NamedTuple('Edge', [('v', int), ('w', int)]) # edge contains two indices to coordinates-array


def compute_partitions(coordinates: np.ndarray, partitions: int) -> List[np.ndarray]:
    """
    Partitions the coordinates into an array of indices. For this matter we use the angles
    between the reference vector of the x-axis.

    :param coordinates: a 2D real array of x- and y-coordinates
    :param partitions: number of partitions
    :return: list of arrays that contain indices of heliostats with regards to the coordinates array
    """
    degrees = np.degrees(np.arctan2(coordinates[1:,1], coordinates[1:,0]))
    indices = degrees.argsort()
    padding = (-len(indices))%partitions
    L = np.split(np.concatenate((indices,np.ones(padding)*-1)),partitions) # padding value: -1
    # remove padding from last partition and increase index
    return [np.delete(a, np.where(a == -1)) + 1 for a in L]

In [9]:
from typing import List

import numpy as np


def equal_coords(M, start_l, end_l, start_r, end_r):
    return np.all(M[:, start_l:end_l] == M[:,start_r:end_r], axis=1)


def disjoint_region(M, i, j, k, l):
    return np.all(np.maximum(M[:,i], M[:,j]) < np.minimum(M[:,k], M[:,l]))


def orientation(M, px, py, qx, qy, rx, ry) -> np.ndarray:
    val = (M[:,qy]-M[:,py]) * (M[:,rx]-M[:,qx]) - (M[:,qx]-M[:,px]) * (M[:,ry]-M[:,qy])
    np.place(val, val > 0, 1)
    np.place(val, val < 0, 2)
    return val


def on_segment(M, px, py, qx, qy, rx, ry) -> np.ndarray:
    return np.logical_and.reduce((M[:,qx] <= np.maximum(M[:,px], M[:,rx]),
                                  M[:,qx] >= np.minimum(M[:,px], M[:,rx]),
                                  M[:,qy] <= np.maximum(M[:,py], M[:,ry]),
                                  M[:,qy] >= np.minimum(M[:,py], M[:,ry])))


def do_intersect(edge: np.ndarray, edge_set: List[np.ndarray], partition_indices: List[bool] = []) -> bool:
    """
    :param edge: array of size (1,4)
    :param edge_set: array of shape (N,4)
    :param partition_indices: boolean list that selects subset of edge_set
    """
    if not partition_indices:
        edge_set = np.vstack(edge_set)
    else:
        edge_set = np.vstack(compress(edge_set, partition_indices))

    # NOTE: ugly hack
    if len(edge_set) == 0:
        return False

    M = np.concatenate((np.repeat(edge, len(edge_set), axis=0), edge_set), axis=1)

    # obtain masking of incident edges
    incident_mask = np.logical_or.reduce((equal_coords(M,0,2,4,6),
                                          equal_coords(M,0,2,6,8),
                                          equal_coords(M,2,4,4,6),
                                          equal_coords(M,2,4,6,8)))
    M = M[np.logical_not(incident_mask)]
    if M.size == 0:
        return False

    o1 = orientation(M,0,1,2,3,4,5)
    o2 = orientation(M,0,1,2,3,6,7)
    o3 = orientation(M,4,5,6,7,0,1)
    o4 = orientation(M,4,5,6,7,2,3)

    if np.logical_and(o1 != o2, o3 != o4).any():
        return True

    if np.logical_and(o1 == 0, on_segment(M,0,1,4,5,2,3)).any():
        return True
    if np.logical_and(o2 == 0, on_segment(M,0,1,6,7,2,3)).any():
        return True
    if np.logical_and(o3 == 0, on_segment(M,4,5,0,1,6,7)).any():
        return True
    if np.logical_and(o4 == 0, on_segment(M,4,5,2,3,6,7)).any():
        return True
    return False


In [10]:
from itertools import chain

class EdgeSolution:
    def __init__(self, partitions: int) -> None:
        """
        :param partitions: number of edge partitions
        """
        self.edges= [[] for _ in range(partitions)] # type: List[List[Edge]]
        self.edges_coords = [np.empty((0,4)) for _ in range(partitions)]
        self.edge_costs = {} # type: Dict[Edge,int]

    def add_edge(self, edge: Edge, edge_cost: float, partition: int, coordinates: np.ndarray):
        self.edges[partition].append(edge)
        self.edge_costs[edge] = edge_cost

        edges_coords = np.array([[coordinates[edge.v][0], coordinates[edge.v][1],
                                 coordinates[edge.w][0], coordinates[edge.w][1]]])
        self.edges_coords[partition] = np.vstack((self.edges_coords[partition], edges_coords))

    def remove_edge(self, edge: Edge, partition: int, coordinates: np.ndarray):
        self.edges[partition].remove(edge)
        del self.edge_costs[edge]

        edges_coords = np.array([[coordinates[edge.v][0], coordinates[edge.v][1],
                                 coordinates[edge.w][0], coordinates[edge.w][1]]])
        delete_index = np.where(np.all(edges_coords == self.edges_coords[partition], axis=1))
        self.edges_coords[partition] = np.delete(self.edges_coords[partition], delete_index, axis=0)

    def intersects(self, edges_coords: np.ndarray) -> bool:
        """
        Checks whether a given edge that is represented by its coordinates is intersecting
        with our solution edges.

        :param edges_coords: a numpy array that holds both x- and y-coordinates
        :returns: True if edges_coords intersects with solution edges
        """
        return is_edge_intersecting(edges_coords, self.edges_coords)

    def cost(self, distances: np.ndarray):
        return sum(distances[e.v,e.w]*self.edge_costs[e] for e in chain.from_iterable(self.edges))


class EdgeVertexSolution(EdgeSolution):
    def __init__(self, n: int, partitions: int):
        super().__init__(partitions)
        self.degrees = {i:0 for i in range(n)} # type: Dict[int,int]

    def add_edge(self, edge: Edge, edge_cost: float, partition: int, coordinates: np.ndarray):
        super().add_edge(edge, edge_cost, partition, coordinates)
        self.degrees[edge.v] += 1
        self.degrees[edge.w] += 1

    def remove_edge(self, edge: Edge, partition: int, coordinates: np.ndarray):
        super().remove_edge(edge, partition, coordinates)
        self.degrees[edge.v] -= 1
        self.degrees[edge.w] -= 1

    def cost(self, distances: np.ndarray):
        cost = super().cost(distances)
        return cost + sum(self._switch_cost(d) for d in self.degrees.values()) - self._switch_cost(self.degrees[0])

    def _switch_cost(self, degree: int) -> float:
        """Note: Hard-coded values for quick and dirty development"""
        if degree <= 2:
            return 100
        elif degree >= 3 and degree <= 9:
            return 800
        elif degree >= 10 and degree <= 17:
            return 1500
        else:
            raise ValueError('Degree is too high. No cost found.')


In [196]:
def a(arr: np.ndarray):
    """
    Works faster than np.roll(arr, -1)
    """
    return arr[1:] + arr[:1]

In [198]:
import sys
from itertools import product
from typing import Dict, List, Set, NamedTuple

import numpy as np
from recordclass import RecordClass
from scipy.spatial.distance import cdist


# TODO: document these NamedTuples
Candidate = RecordClass('Candidate', [('cost', float),
                                      ('edges', List[Edge]),
                                      ('vertex', int),
                                      ('index', int)])
HamiltonState = RecordClass('HamiltonState', [('permutation', List[int]),
                                              ('unvisited_heliostats', Set[int])])


def shift_left(arr: np.ndarray):
    """
    Works faster than np.roll(arr, -1)
    """
    return arr[1:] + arr[:1]
    
    
class Hamilton:
    """
    We compute Hamiltonian paths for each partition that are initial solutions for the
    data cabling.
    """
    __slots__ = ('coordinates', 'cable_cost', 'edge_costs', 'partitions', 'solution', 'current_state')

    def __init__(self, coordinates: np.ndarray, cable_cost: float, partitions: int) -> None:
        """
        :param coordinates: coordinates vertices
        :param cable: cost for hamiltonian path
        :param partitions: number of partitions
        """
        self.coordinates = coordinates
        self.cable_cost = cable_cost
        self.edge_costs = cdist(self.coordinates, self.coordinates) * cable_cost # only consider glass fiber cables!
        self.partitions = partitions
        self.solution = EdgeVertexSolution(self.coordinates.shape[0], partitions)
        self.current_state = HamiltonState(None, None)

    def compute(self) -> None:
        """
        Compute Hamiltonian paths for each partition.
        """
        for i, p in enumerate(compute_partitions(coordinates=self.coordinates, partitions=self.partitions)):
            self._compute_hamilton(i, p)

    def _compute_hamilton(self, partition: int, partition_indices: np.ndarray) -> None:
        """
        Computes for a partition of indices a valid solution regarding planarity.

        :param partition_indices: indices of heliostats that belong together in a partition
        :return: edges and edge_coords that form a Hamiltonian path
        """
        self.current_state._replace(permutation=[0])
        self.current_state._replace(unvisited_heliostats=set(partition_indices.astype(int).flat))

        while len(self.solution.edges[partition]) < partition_indices.shape[0]:
            best_candidate = min(self._compute_candidates())
            if len(best_candidate.edges) == 2:
                self._insert_between(best_candidate, partition)
            else:
                self._insert_last(best_candidate, partition)

    def _compute_candidates(self) -> Candidate:
        """
        Compute cost and edges for a vertex and index position of permutation. In other words
        for a vertex v we compute its minimum cost of insertion into the current permutation
        of our Hamiltonian path. This can either be an insertion and appending to the permutation.

        :param vertex: vertex to be added to permutation
        :param index: index of insertion/appending
        """
        for vertex in self.current_state.unvisited_heliostats:
            shifted_perm = shift_left(self.current_state.permutation)
            unvisited = [vertex]*len(self.current_state.permutation)

            costs_edge_two = self.edge_costs[ unvisited, shifted_perm]
            costs_edge_two[-1] = 0
            adding_costs = self.edge_costs[ unvisited, self.current_state.permutation] + costs_edge_two

            cost_permutation_edges = self.edge_costs[self.current_state.permutation, shifted_perm]
            cost_permutation_edges[-1] = 0
            index = np.argmin(adding_costs - cost_permutation_edges)

            if index < len(self.current_state.permutation)-1:
                e1 = Edge(self.current_state.permutation[index], vertex)
                e2 = Edge(vertex, self.current_state.permutation[index+1])
                d = Edge(self.current_state.permutation[index], self.current_state.permutation[index+1])

                cost = self.edge_costs[e1.v, e1.w] + self.edge_costs[e2.v, e2.w] - self.edge_costs[d.v, d.w]
                yield Candidate(cost=cost, edges=[e1,e2], vertex=vertex, index=index)
            else:
                e = Edge(self.current_state.permutation[index], vertex)
                yield Candidate(cost=self.edge_costs[e.v, e.w], edges=[e], vertex=vertex, index=index)

    def _insert_edge(self, edge: Edge, partition: int) -> None:
        """
        Adds edge to solution partition
        """
        self.solution.add_edge(edge=edge,
                               edge_cost=self.cable_cost,
                               partition=partition,
                               coordinates=self.coordinates)
        # update current unvisited heliostats set
        for vertex in edge:
            if vertex in self.current_state.unvisited_heliostats:
                self.current_state.unvisited_heliostats.remove(vertex)

    def _insert_between(self, candidate: Candidate, partition: int) -> None:
        # remove old edge
        index = candidate.index
        old_edge = Edge(self.current_state.permutation[index], self.current_state.permutation[index+1])
        self.solution.remove_edge(old_edge, partition, self.coordinates)
        # add new edges
        self._insert_edge(candidate.edges[0], partition)
        self._insert_edge(candidate.edges[1], partition)
        # update current permutation
        self.current_state.permutation.insert(candidate.index+1, candidate.vertex)

    def _insert_last(self, candidate: Candidate, partition: int) -> None:
        self._insert_edge(candidate.edges[0], partition)
        self.current_state.permutation.append(candidate.vertex)


In [199]:
coordinates = np.loadtxt('../data/instances/PS10.csv', delimiter=';')
coordinates = np.vstack((np.array([0,0]), coordinates))

In [200]:
#coordinates = coordinates[:450]

## hamilton

In [201]:
%load_ext line_profiler

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


In [202]:
ham = Hamilton(coordinates, 14, 1)

In [203]:
%lprun -f ham._compute_candidates ham.compute()

In [159]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm

def plot_solution_data(coordinates: np.ndarray,
                       output: str,
                       edges: List[List[Edge]],
                       value: float,
                       edge_cost: float,
                       partitions: int) -> None:
    plt.figure(figsize=(16,16), dpi=350)
    plt.axis('equal')

    coordinates_arr = np.array([tuple(x) for x in coordinates])
    for i,all_edges in enumerate(edges):
        edges_arr = np.array([tuple(list(x)) for x in all_edges])

        x = coordinates_arr[:,0].flatten()
        y = coordinates_arr[:,1].flatten()

        norm = matplotlib.colors.Normalize(vmin=0, vmax=len(edges), clip=True)
        mapper = cm.ScalarMappable(norm=norm, cmap=cm.cool)

        plt.plot(x[edges_arr.T], y[edges_arr.T], linestyle='-', color=mapper.to_rgba(i),
                 markerfacecolor='red', marker='o')
    plt.title("Cable cost/m: {}€ \nPartitions: {}\nTotal costs: {:0,.2f}€".format(edge_cost, partitions, value))
    plt.scatter(coordinates_arr[:,0], coordinates_arr[:,1], color='red', marker='o')
    plt.savefig(output, dpi=300)
    plt.close()


In [160]:
q = cdist(coordinates, coordinates)

In [161]:
plot_solution_data(coordinates, '/home/duc/a.png', ham.solution.edges, ham.solution.cost(q), 14, 5)

# numpy cost check

In [184]:
%timeit np.concatenate([d[1:],d[:1]])

926 ns ± 7.46 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [177]:
d = np.array([1,2,3,4])

In [185]:
%timeit np.roll(d, -1)

10.2 µs ± 129 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)
